In [11]:
# ============================================================================
# File: tokenizers.py
# Multimodal Tokenizers for Video, Audio, and Text
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoProcessor, AutoImageProcessor
import numpy as np
import librosa


class VideoTokenizer(nn.Module):
    """
    Video Tokenizer using VideoMAE + VQ-VAE
    Converts video frames to discrete tokens
    """
    def __init__(self, vocab_size=8192, codebook_dim=768):
        super().__init__()
        from transformers import VideoMAEModel
        
        self.encoder = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
        self.processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
        
        # Freeze encoder
        for param in self.encoder.parameters():
            param.requires_grad = False
        
        # VQ-VAE codebook
        self.vocab_size = vocab_size
        self.codebook_dim = codebook_dim
        self.codebook = nn.Embedding(vocab_size, codebook_dim)
        
        # Projection to codebook dimension
        self.proj = nn.Linear(768, codebook_dim)
        
        # Initialize codebook
        self.codebook.weight.data.uniform_(-1.0 / vocab_size, 1.0 / vocab_size)
        
    def forward(self, pixel_values):
        """
        Args:
            pixel_values: (B, T, C, H, W) or preprocessed
        Returns:
            tokens: (B, T) discrete token indices
        """
        with torch.no_grad():
            outputs = self.encoder(pixel_values)
            features = outputs.last_hidden_state.mean(dim=1)  # (B, 768)
        
        # Project to codebook dimension
        z = self.proj(features)  # (B, codebook_dim)
        
        # Quantize
        distances = torch.cdist(z, self.codebook.weight)  # (B, vocab_size)
        tokens = torch.argmin(distances, dim=-1)  # (B,)
        
        return tokens.unsqueeze(1)  # (B, 1) for single frame per sample
    
    def encode_frames(self, frames):
        """
        Encode multiple frames
        Args:
            frames: list of (C, H, W) or (H, W, C) numpy arrays
        Returns:
            tokens: (1, n_frames) 
        """
        # Process frames
        inputs = self.processor(frames, return_tensors="pt")
        pixel_values = inputs['pixel_values']
        
        tokens = []
        with torch.no_grad():
            for i in range(0, len(frames), 16):  # Process in chunks of 16
                chunk = pixel_values[i:min(i+16, len(frames))]
                chunk_tokens = self.forward(chunk)
                tokens.append(chunk_tokens)
        
        return torch.cat(tokens, dim=1)  # (1, n_frames)


class SpeechTokenizer(nn.Module):
    """
    Speech Tokenizer using HuBERT + K-means clustering
    Converts audio to discrete tokens
    """
    def __init__(self, vocab_size=1024):
        super().__init__()
        from transformers import HubertModel
        
        self.encoder = HubertModel.from_pretrained("facebook/hubert-base-ls960")
        self.processor = AutoProcessor.from_pretrained("facebook/hubert-base-ls960")
        
        # Freeze encoder
        for param in self.encoder.parameters():
            param.requires_grad = False
        
        self.vocab_size = vocab_size
        # K-means centers (learned during preprocessing)
        self.register_buffer('kmeans_centers', torch.randn(vocab_size, 768))
        
    def forward(self, input_values):
        """
        Args:
            input_values: (B, L) audio waveform
        Returns:
            tokens: (B, T) discrete token indices
        """
        with torch.no_grad():
            outputs = self.encoder(input_values)
            features = outputs.last_hidden_state  # (B, T, 768)
        
        # Quantize using k-means
        B, T, D = features.shape
        features_flat = features.reshape(-1, D)
        
        distances = torch.cdist(features_flat, self.kmeans_centers)
        tokens = torch.argmin(distances, dim=-1)
        tokens = tokens.reshape(B, T)
        
        # Downsample by factor of 2 to reduce sequence length
        tokens = tokens[:, ::2]
        
        return tokens
    
    def encode_audio(self, audio_path, sr=16000):
        """
        Encode audio file
        Args:
            audio_path: path to audio file
            sr: sampling rate
        Returns:
            tokens: (1, T)
        """
        audio, _ = librosa.load(audio_path, sr=sr)
        inputs = self.processor(audio, sampling_rate=sr, return_tensors="pt")
        return self.forward(inputs.input_values)

In [2]:
# ============================================================================
# File: model.py
# Multimodal Token LLM Model
# ============================================================================

import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Config


class MultimodalTokenLLM(nn.Module):
    """
    Multimodal Token LLM for Turn-Taking Prediction
    
    Architecture:
    1. Tokenize video, audio, text into discrete tokens
    2. Embed tokens into LLM space
    3. Concatenate: [VID][AUD][TXT][PREDICT]
    4. Pass through LLM
    5. Classify final token: keep/turn/backchannel
    """
    def __init__(
        self, 
        llm_model_name="gpt2",
        video_vocab_size=8192,
        audio_vocab_size=1024,
        num_classes=3,
        freeze_llm=False,
        use_pretrained_tokenizers=True
    ):
        super().__init__()
        
        # Load or initialize tokenizers
        if use_pretrained_tokenizers:
            self.video_tokenizer = VideoTokenizer(vocab_size=video_vocab_size)
            self.audio_tokenizer = SpeechTokenizer(vocab_size=audio_vocab_size)
        else:
            self.video_tokenizer = None
            self.audio_tokenizer = None
        
        self.text_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
        self.text_tokenizer.pad_token = self.text_tokenizer.eos_token
        
        # Load LLM
        self.llm = GPT2LMHeadModel.from_pretrained(llm_model_name)
        self.hidden_dim = self.llm.config.n_embd  # 768 for GPT2
        
        if freeze_llm:
            for param in self.llm.parameters():
                param.requires_grad = False
        
        # Modality embeddings
        self.video_embedding = nn.Embedding(video_vocab_size, self.hidden_dim)
        self.audio_embedding = nn.Embedding(audio_vocab_size, self.hidden_dim)
        
        # Special tokens
        self.predict_token = nn.Parameter(torch.randn(1, 1, self.hidden_dim))
        
        # Modality type embeddings (helps LLM distinguish modalities)
        self.modality_type_embedding = nn.Embedding(4, self.hidden_dim)  # video, audio, text, predict
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_dim // 2, num_classes)
        )
        
        # Initialize embeddings
        nn.init.normal_(self.video_embedding.weight, std=0.02)
        nn.init.normal_(self.audio_embedding.weight, std=0.02)
        nn.init.normal_(self.modality_type_embedding.weight, std=0.02)
        
    def forward(self, video_tokens=None, audio_tokens=None, text_ids=None, 
                attention_mask=None, return_embeddings=False):
        """
        Args:
            video_tokens: (B, T_v) discrete video tokens
            audio_tokens: (B, T_a) discrete audio tokens  
            text_ids: (B, T_t) text token ids
            attention_mask: (B, total_length) attention mask
        Returns:
            logits: (B, 3) classification logits
        """
        B = video_tokens.shape[0] if video_tokens is not None else \
            audio_tokens.shape[0] if audio_tokens is not None else text_ids.shape[0]
        device = video_tokens.device if video_tokens is not None else \
                 audio_tokens.device if audio_tokens is not None else text_ids.device
        
        embeddings_list = []
        modality_ids_list = []
        
        # Video embeddings
        if video_tokens is not None:
            video_embeds = self.video_embedding(video_tokens)  # (B, T_v, H)
            embeddings_list.append(video_embeds)
            modality_ids_list.append(torch.zeros(video_tokens.shape, dtype=torch.long, device=device))
        
        # Audio embeddings
        if audio_tokens is not None:
            audio_embeds = self.audio_embedding(audio_tokens)  # (B, T_a, H)
            embeddings_list.append(audio_embeds)
            modality_ids_list.append(torch.ones(audio_tokens.shape, dtype=torch.long, device=device))
        
        # Text embeddings
        if text_ids is not None:
            text_embeds = self.llm.transformer.wte(text_ids)  # (B, T_t, H)
            embeddings_list.append(text_embeds)
            modality_ids_list.append(torch.full(text_ids.shape, 2, dtype=torch.long, device=device))
        
        # Predict token
        predict_embed = self.predict_token.expand(B, -1, -1)  # (B, 1, H)
        embeddings_list.append(predict_embed)
        modality_ids_list.append(torch.full((B, 1), 3, dtype=torch.long, device=device))
        
        # Concatenate all embeddings
        sequence_embeds = torch.cat(embeddings_list, dim=1)  # (B, total_length, H)
        modality_ids = torch.cat(modality_ids_list, dim=1)  # (B, total_length)
        
        # Add modality type embeddings
        modality_type_embeds = self.modality_type_embedding(modality_ids)
        sequence_embeds = sequence_embeds + modality_type_embeds
        
        # Create attention mask if not provided
        if attention_mask is None:
            attention_mask = torch.ones(sequence_embeds.shape[:2], device=device)
        
        if return_embeddings:
            return sequence_embeds
        
        # Pass through LLM
        outputs = self.llm(
            inputs_embeds=sequence_embeds,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        # Get final token representation (the PREDICT token)
        final_hidden = outputs.hidden_states[-1][:, -1, :]  # (B, H)
        
        # Classify
        logits = self.classifier(final_hidden)  # (B, 3)
        
        return logits

In [ ]:



# ============================================================================
# File: dataloader_token.py
# DataLoader for Tokenized Multimodal Data
# ============================================================================

import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import librosa
from tqdm import tqdm


class MultimodalTokenDataset(Dataset):
    """
    Dataset that loads and tokenizes video, audio, text on-the-fly
    """
    def __init__(
        self, 
        data_root, 
        split, 
        video_tokenizer,
        audio_tokenizer,
        text_tokenizer,
        n_video_frames=16,
        max_audio_length=80000,
        max_text_length=128
    ):
        assert split in ["train", "val", "test"]
        
        self.data_root = data_root
        self.split = split
        self.video_tokenizer = video_tokenizer
        self.audio_tokenizer = audio_tokenizer
        self.text_tokenizer = text_tokenizer
        self.n_video_frames = n_video_frames
        self.max_audio_length = max_audio_length
        self.max_text_length = max_text_length
        self.sampling_rate = 16000
        
        # Load metadata
        df = pd.read_csv(os.path.join(data_root, f"{split}.csv"), sep="\t")
        self.sentence_ids = df["sentence_id"].values
        self.texts = df["text"].values
        self.labels = df["label"].values
        
        self.audio_root = os.path.join(data_root, "audio")
        self.video_root = os.path.join(data_root, "video")
        
    def __len__(self):
        return len(self.sentence_ids)
    
    def load_video_frames(self, video_id, sentence_id):
        """Load video frames"""
        frames = []
        for i in range(self.n_video_frames):
            img_path = os.path.join(self.video_root, video_id, sentence_id, f"{i}.jpg")
            if not os.path.exists(img_path):
                break
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                frames.append(img)
        
        # Pad if needed
        while len(frames) < self.n_video_frames:
            if len(frames) > 0:
                frames.append(frames[-1])
            else:
                frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
        
        return frames[:self.n_video_frames]
    
    def load_audio(self, audio_path):
        """Load audio"""
        try:
            audio, _ = librosa.load(audio_path, sr=self.sampling_rate)
            # Truncate or pad
            if len(audio) > self.max_audio_length:
                audio = audio[:self.max_audio_length]
            else:
                audio = np.pad(audio, (0, self.max_audio_length - len(audio)))
            return audio
        except:
            return np.zeros(self.max_audio_length)
    
    def __getitem__(self, idx):
        sentence_id = self.sentence_ids[idx]
        text = self.texts[idx]
        label = self.labels[idx]
        
        video_id = "_".join(sentence_id.split("_")[:-2])
        
        # Load raw data
        frames = self.load_video_frames(video_id, sentence_id)
        audio_path = os.path.join(self.audio_root, video_id, f"{sentence_id}.mp3")
        audio = self.load_audio(audio_path)
        
        # Tokenize
        with torch.no_grad():
            # Video tokens
            video_inputs = self.video_tokenizer.processor(frames, return_tensors="pt")
            video_tokens = self.video_tokenizer(video_inputs['pixel_values'])
            
            # Audio tokens
            audio_inputs = self.audio_tokenizer.processor(
                audio, sampling_rate=self.sampling_rate, return_tensors="pt"
            )
            audio_tokens = self.audio_tokenizer(audio_inputs.input_values)
            
            # Text tokens
            text_inputs = self.text_tokenizer(
                text,
                max_length=self.max_text_length,
                padding='max_length',
                truncation=True,
                return_tensors="pt"
            )
        
        return {
            'video_tokens': video_tokens.squeeze(0),  # (T_v,)
            'audio_tokens': audio_tokens.squeeze(0),  # (T_a,)
            'text_ids': text_inputs['input_ids'].squeeze(0),  # (T_t,)
            'text_attention_mask': text_inputs['attention_mask'].squeeze(0),
            'label': label
        }


def collate_fn_token(batch):
    """Custom collate function for variable-length sequences"""
    video_tokens = torch.stack([x['video_tokens'] for x in batch])
    audio_tokens = torch.nn.utils.rnn.pad_sequence(
        [x['audio_tokens'] for x in batch], 
        batch_first=True, 
        padding_value=0
    )
    text_ids = torch.stack([x['text_ids'] for x in batch])
    text_attention_mask = torch.stack([x['text_attention_mask'] for x in batch])
    labels = torch.tensor([x['label'] for x in batch])
    
    return video_tokens, audio_tokens, text_ids, text_attention_mask, labels

In [10]:
import sys

if 'ipykernel' in sys.modules:
    sys.argv = [sys.argv[0]]
# ============================================================================
# File: train_token_llm.py
# Training Script for Multimodal Token LLM
# ============================================================================

import os
import time
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Assuming all classes are imported from above modules


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_root", type=str, default="dataset/")
    parser.add_argument("--device", type=str, default="cuda")
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--n_workers", type=int, default=4)
    parser.add_argument("--n_epochs", type=int, default=20)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--llm_lr", type=float, default=1e-5)
    parser.add_argument("--freeze_llm", action="store_true", default=False)
    parser.add_argument("--log_dir", type=str, default="log_token_llm")
    parser.add_argument("--save_every", type=int, default=5)
    #return parser.parse_args()
    args, unknown = parser.parse_known_args()
    return args

def cal_metrics(all_labels, all_preds):
    accuracy = accuracy_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    return accuracy, recall, f1, precision


def idx2label(idx):
    return ["keep", "turn", "bc"][idx]


def main():
    args = parse_args()
    
    # Create log directory
    os.makedirs(args.log_dir, exist_ok=True)
    time_str = time.strftime("%Y-%m-%d_%H-%M-%S")
    task_dir = os.path.join(args.log_dir, f"{time_str}_token_llm")
    writer = SummaryWriter(log_dir=task_dir)
    
    # Initialize tokenizers
    print("Initializing tokenizers...")
    video_tokenizer = VideoTokenizer()
    audio_tokenizer = SpeechTokenizer()
    text_tokenizer = AutoTokenizer.from_pretrained("gpt2")
    text_tokenizer.pad_token = text_tokenizer.eos_token
    
    # Load datasets
    print("Loading datasets...")
    train_dataset = MultimodalTokenDataset(
        args.data_root, "train",
        video_tokenizer, audio_tokenizer, text_tokenizer
    )
    val_dataset = MultimodalTokenDataset(
        args.data_root, "val",
        video_tokenizer, audio_tokenizer, text_tokenizer
    )
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=args.batch_size, 
        shuffle=True,
        collate_fn=collate_fn_token,
        num_workers=args.n_workers
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        collate_fn=collate_fn_token,
        num_workers=args.n_workers
    )
    
    # Initialize model
    print("Initializing model...")
    model = MultimodalTokenLLM(
        freeze_llm=args.freeze_llm,
        use_pretrained_tokenizers=False  # We pass tokenizers separately
    )
    model.video_tokenizer = video_tokenizer
    model.audio_tokenizer = audio_tokenizer
    model = model.to(args.device)
    
    # Optimizer with different learning rates
    llm_params = list(model.llm.parameters())
    other_params = [p for n, p in model.named_parameters() if 'llm' not in n]
    
    optimizer = optim.AdamW([
        {'params': other_params, 'lr': args.lr},
        {'params': llm_params, 'lr': args.llm_lr}
    ])
    
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.n_epochs)
    
    # Training loop
    print("Starting training...")
    best_val_acc = 0.0
    
    for epoch in range(args.n_epochs):
        # Train
        model.train()
        train_loss = 0.0
        all_labels, all_preds = [], []
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{args.n_epochs} [Train]")
        for video_tokens, audio_tokens, text_ids, text_mask, labels in train_bar:
            video_tokens = video_tokens.to(args.device)
            audio_tokens = audio_tokens.to(args.device)
            text_ids = text_ids.to(args.device)
            labels = labels.to(args.device)
            
            optimizer.zero_grad()
            
            logits = model(
                video_tokens=video_tokens,
                audio_tokens=audio_tokens,
                text_ids=text_ids
            )
            
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            preds = logits.argmax(dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            
            train_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        train_loss /= len(train_loader)
        train_acc, train_recall, train_f1, train_prec = cal_metrics(all_labels, all_preds)
        
        # Log training metrics
        writer.add_scalar("train/loss", train_loss, epoch)
        writer.add_scalar("train/accuracy", train_acc, epoch)
        
        # Validation
        model.eval()
        val_loss = 0.0
        all_labels, all_preds = [], []
        
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{args.n_epochs} [Val]")
        with torch.no_grad():
            for video_tokens, audio_tokens, text_ids, text_mask, labels in val_bar:
                video_tokens = video_tokens.to(args.device)
                audio_tokens = audio_tokens.to(args.device)
                text_ids = text_ids.to(args.device)
                labels = labels.to(args.device)
                
                logits = model(
                    video_tokens=video_tokens,
                    audio_tokens=audio_tokens,
                    text_ids=text_ids
                )
                
                loss = criterion(logits, labels)
                val_loss += loss.item()
                
                preds = logits.argmax(dim=1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc, val_recall, val_f1, val_prec = cal_metrics(all_labels, all_preds)
        
        # Log validation metrics
        writer.add_scalar("val/loss", val_loss, epoch)
        writer.add_scalar("val/accuracy", val_acc, epoch)
        
        print(f"\nEpoch {epoch+1}/{args.n_epochs}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_path = os.path.join(task_dir, "best_model.pt")
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model with val_acc: {val_acc:.4f}")
        
        # Save checkpoint
        if (epoch + 1) % args.save_every == 0:
            save_path = os.path.join(task_dir, f"checkpoint_epoch_{epoch+1}.pt")
            torch.save(model.state_dict(), save_path)
        
        scheduler.step()
    
    writer.close()
    print(f"Training complete! Best val accuracy: {best_val_acc:.4f}")


if __name__ == "__main__":
    main()

Initializing tokenizers...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:

# ============================================================================
# File: inference_token_llm.py
# Inference Script
# ============================================================================

import argparse
import torch
import cv2
import librosa
import numpy as np
from tqdm import tqdm


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--video_path", type=str, required=True)
    parser.add_argument("--ckpt_path", type=str, required=True)
    parser.add_argument("--device", type=str, default="cuda")
    parser.add_argument("--n_frames", type=int, default=16)
    parser.add_argument("--samplerate", type=int, default=16000)
    return parser.parse_args()


def idx2label(idx):
    return {0: "keep", 1: "turn-taking", 2: "backchannel"}[idx]


def read_video(video_path, n_frames=16):
    """Read video and extract frames"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    
    cap.release()
    return frames, fps


def main():
    args = parse_args()
    
    # Load model
    print("Loading model...")
    model = MultimodalTokenLLM()
    model.load_state_dict(torch.load(args.ckpt_path))
    model = model.to(args.device)
    model.eval()
    
    # Load video and audio
    print("Loading video and audio...")
    frames, fps = read_video(args.video_path, args.n_frames)
    audio, _ = librosa.load(args.video_path, sr=args.samplerate)
    
    # Simple transcription (you can use WhisperX here)
    text = "This is a placeholder text. Use WhisperX for actual transcription."
    
    # Tokenize
    print("Tokenizing...")
    with torch.no_grad():
        # Video
        video_inputs = model.video_tokenizer.processor(frames[:args.n_frames], return_tensors="pt")
        video_tokens = model.video_tokenizer(video_inputs['pixel_values'].to(args.device))
        
        # Audio
        audio_inputs = model.audio_tokenizer.processor(audio[:80000], sampling_rate=args.samplerate, return_tensors="pt")
        audio_tokens = model.audio_tokenizer(audio_inputs.input_values.to(args.device))
        
        # Text
        text_inputs = model.text_tokenizer(text, return_tensors="pt", max_length=128, truncation=True)
        text_ids = text_inputs['input_ids'].to(args.device)
        
        # Predict
        logits = model(
            video_tokens=video_tokens,
            audio_tokens=audio_tokens,
            text_ids=text_ids
        )
        
        probs = torch.softmax(logits, dim=-1)
        pred = logits.argmax(dim=-1).item()
        
    print("\n" + "="*50)
    print("PREDICTION RESULT")
    print("="*50)
    print(f"Action: {idx2label(pred)}")
    print(f"Probabilities:")
    print(f"  Keep: {probs[0, 0].item():.4f}")
    print(f"  Turn: {probs[0, 1].item():.4f}")
    print(f"  Backchannel: {probs[0, 2].item():.4f}")
    print("="*50)


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================================
# File: evaluate_token_llm.py
# Evaluation Script
# ============================================================================

import argparse
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_root", type=str, default="dataset/")
    parser.add_argument("--ckpt_path", type=str, required=True)
    parser.add_argument("--device", type=str, default="cuda")
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--split", type=str, default="test", choices=["train", "val", "test"])
    parser.add_argument("--output_dir", type=str, default="eval_results")
    #return parser.parse_args()
    args, unknown = parser.parse_known_args()
    return args

def idx2label(idx):
    return ["keep", "turn", "bc"][idx]


def plot_confusion_matrix(cm, output_path):
    """Plot and save confusion matrix"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=['Keep', 'Turn', 'BC'],
        yticklabels=['Keep', 'Turn', 'BC']
    )
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def main():
    args = parse_args()
    os.makedirs(args.output_dir, exist_ok=True)
    
    # Load model
    print("Loading model...")
    model = MultimodalTokenLLM()
    model.load_state_dict(torch.load(args.ckpt_path, map_location=args.device))
    model = model.to(args.device)
    model.eval()
    
    # Initialize tokenizers
    print("Initializing tokenizers...")
    video_tokenizer = VideoTokenizer()
    audio_tokenizer = SpeechTokenizer()
    text_tokenizer = AutoTokenizer.from_pretrained("gpt2")
    text_tokenizer.pad_token = text_tokenizer.eos_token
    
    # Load dataset
    print(f"Loading {args.split} dataset...")
    test_dataset = MultimodalTokenDataset(
        args.data_root, args.split,
        video_tokenizer, audio_tokenizer, text_tokenizer
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        collate_fn=collate_fn_token,
        num_workers=4
    )
    
    # Evaluate
    print("Evaluating...")
    all_labels = []
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for video_tokens, audio_tokens, text_ids, text_mask, labels in tqdm(test_loader):
            video_tokens = video_tokens.to(args.device)
            audio_tokens = audio_tokens.to(args.device)
            text_ids = text_ids.to(args.device)
            
            logits = model(
                video_tokens=video_tokens,
                audio_tokens=audio_tokens,
                text_ids=text_ids
            )
            
            probs = torch.softmax(logits, dim=-1)
            preds = logits.argmax(dim=-1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    
    # Per-class metrics
    f1_per_class = f1_score(all_labels, all_preds, average=None)
    precision_per_class = precision_score(all_labels, all_preds, average=None)
    recall_per_class = recall_score(all_labels, all_preds, average=None)
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    # Print results
    print("\n" + "="*70)
    print("EVALUATION RESULTS")
    print("="*70)
    print(f"Overall Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {f1_macro:.4f}")
    print(f"Weighted F1: {f1_weighted:.4f}")
    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall: {recall:.4f}")
    print("\nPer-Class Metrics:")
    print("-"*70)
    for i, label in enumerate(['Keep', 'Turn', 'Backchannel']):
        print(f"{label:12s} - Precision: {precision_per_class[i]:.4f}, "
              f"Recall: {recall_per_class[i]:.4f}, F1: {f1_per_class[i]:.4f}")
    print("="*70)
    
    # Detailed classification report
    print("\nDetailed Classification Report:")
    print(classification_report(
        all_labels, all_preds,
        target_names=['Keep', 'Turn', 'Backchannel']
    ))
    
    # Save confusion matrix
    cm_path = os.path.join(args.output_dir, "confusion_matrix.png")
    plot_confusion_matrix(cm, cm_path)
    print(f"\nConfusion matrix saved to: {cm_path}")
    
    # Save results to file
    results_path = os.path.join(args.output_dir, "evaluation_results.txt")
    with open(results_path, 'w') as f:
        f.write("="*70 + "\n")
        f.write("EVALUATION RESULTS\n")
        f.write("="*70 + "\n")
        f.write(f"Overall Accuracy: {accuracy:.4f}\n")
        f.write(f"Macro F1: {f1_macro:.4f}\n")
        f.write(f"Weighted F1: {f1_weighted:.4f}\n")
        f.write(f"Macro Precision: {precision:.4f}\n")
        f.write(f"Macro Recall: {recall:.4f}\n\n")
        f.write("Per-Class Metrics:\n")
        f.write("-"*70 + "\n")
        for i, label in enumerate(['Keep', 'Turn', 'Backchannel']):
            f.write(f"{label:12s} - Precision: {precision_per_class[i]:.4f}, "
                   f"Recall: {recall_per_class[i]:.4f}, F1: {f1_per_class[i]:.4f}\n")
        f.write("="*70 + "\n\n")
        f.write("Confusion Matrix:\n")
        f.write(str(cm) + "\n")
    
    print(f"Results saved to: {results_path}")


if __name__ == "__main__":
    main()